# Exercise sheet 3

In this notebook, you will find the skeleton of the code that you need for tasks 1-4. The skeleton includes instructions and hints as comments. Fill in your code where you are instructed to do and also leave brief explanations of your code using comments, where necessary. Run the cells in the given order and make sure that you first run the following cell to import all necessary libraries  

In [101]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Task 1: Long live Attention

Define $X$, $W_Q$, $W_K$, and $W_V$. Remember that you need to convert them to tensors. Print the values.

> NOTE: Remember to round up your results maybe to 1 or 2 decimal places.

In [102]:
# Initialisation

x = [
  [1, 0, 1, 0],
  [0, 2, 0, 2],
  [1, 1, 1, 1]
 ]

w_k = [
  [0, 0, 1],
  [1, 1, 0],
  [0, 1, 0],
  [1, 1, 0]
]

w_q = [
  [1, 0, 1],
  [1, 0, 0],
  [0, 0, 1],
  [0, 1, 1]
]

w_v = [
  [0, 2, 0],
  [0, 3, 0],
  [1, 0, 3],
  [1, 1, 0]
]

# TODO: Convert to tensors (use the tensor function from the torch library)

# Convert to tensors
x   = torch.tensor(x, dtype=torch.float32)
w_k = torch.tensor(w_k, dtype=torch.float32)
w_q = torch.tensor(w_q, dtype=torch.float32)
w_v = torch.tensor(w_v, dtype=torch.float32)


# TODO: Print the values

# Print the values
print("x=\n ", x)
print("w_k= \n", w_k)
print("w_q=\n", w_q)
print("w_v=\n", w_v)


x=
  tensor([[1., 0., 1., 0.],
        [0., 2., 0., 2.],
        [1., 1., 1., 1.]])
w_k= 
 tensor([[0., 0., 1.],
        [1., 1., 0.],
        [0., 1., 0.],
        [1., 1., 0.]])
w_q=
 tensor([[1., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.],
        [0., 1., 1.]])
w_v=
 tensor([[0., 2., 0.],
        [0., 3., 0.],
        [1., 0., 3.],
        [1., 1., 0.]])


Calculate the values of $Q$, $K$, and $V$. Print the values.

In [103]:
# TODO: Calculate Q, K, and V

q = x @ w_q    
k = x @ w_k    
v = x @ w_v    

# TODO: Print the values of Q, K, and V
print("Q =\n", q)
print("K =\n", k)
print("V =\n", v)


Q =
 tensor([[1., 0., 2.],
        [2., 2., 2.],
        [2., 1., 3.]])
K =
 tensor([[0., 1., 1.],
        [4., 4., 0.],
        [2., 3., 1.]])
V =
 tensor([[1., 2., 3.],
        [2., 8., 0.],
        [2., 6., 3.]])


Calculate the attention scores from the queries. Print the attention scores.

*Hint: Think which matrices you need for this.*

In [104]:
# TODO: Calculate the attention scores matrix

attn_scores = q @ k.T / torch.sqrt(torch.tensor(k.shape[1], dtype=torch.float32))

# TODO: Print the matrix

print("Attention Scores =\n", attn_scores)

Attention Scores =
 tensor([[1.1547, 2.3094, 2.3094],
        [2.3094, 9.2376, 6.9282],
        [2.3094, 6.9282, 5.7735]])


Calculate the *softmax* for the attention scores. Print the value.

In [105]:
from torch.nn.functional import softmax

# TODO: Calculate the softmax

attn_scores_softmax = softmax(attn_scores, dim=1)

# TODO: An extra step needs to be performed here. What could it be?
# Hint: Check the notebook again ;)



# TODO: Print the value
print(attn_scores_softmax)


tensor([[1.3613e-01, 4.3194e-01, 4.3194e-01],
        [8.9045e-04, 9.0884e-01, 9.0267e-02],
        [7.4449e-03, 7.5471e-01, 2.3785e-01]])


Calculate the weighted values by multiplying the softmax attention scores with the values. Print the weighted values.

In [106]:
# TODO: Calculate the weighted values (value matrix with attention scores)

weighted_values = attn_scores_softmax @ v

# TODO: Print the weighted values
print("Weighted Values =\n", weighted_values)


Weighted Values =
 tensor([[1.8639, 6.3194, 1.7042],
        [1.9991, 7.8141, 0.2735],
        [1.9926, 7.4796, 0.7359]])


Finally, sum up the weighted values to get your final result. Print the result.

In [107]:
# TODO: Sum up the weighted values

outputs = weighted_values

# TODO: Print the final result

print("Final Output =\n", outputs)

# You may round up the values or leave them as they are.
# Did you get the same results as 1.(b)?



Final Output =
 tensor([[1.8639, 6.3194, 1.7042],
        [1.9991, 7.8141, 0.2735],
        [1.9926, 7.4796, 0.7359]])


## Task 2: MyGPT: Building a GPT-style language model

a) Attention Head

We have $$T \times n_{embedd}$$ size matrices, were row i corresponds to representation of token i. Batchsize $B$ is for training

In [108]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        #headsize is number if multihead attentions
        # Define the key, query and value weights as linear layers.
        # Do not worry about the n_embed, block size and dropout variables
        # we will define them later as hyperparameters.
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Input dimensions
        B,T,C = x.shape # Batch, Time (seq length), Channels (embedding dim) 
        # TODO: compute query and key matrices 
        k = self.key(x)
        q = self.query(x)
        # TODO: compute the normalized similarity scores between query and key
        # Hint: do not forget to add the mask
        wei = q @ k.transpose(-2, -1) * (1.0 / (k.size(-1) ** 0.5))       
        # TODO: Compute the attention weight
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # we add mask for multihead attention
        # Here we pass the attention weight through a dropout layer
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        # TODO: Compute the value matrix 
        v = self.value(x)
        # TODO: perform the weighted aggregation of the values
        out = wei @ v  # (B, T, head_size)
        #print("Head output:", out)
        return out


b) Multihead Attention

In [109]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        # The list of our attention heads
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # The output weight matrix 
        self.proj = nn.Linear(n_embd, n_embd)
        # A dropout layer
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
       # TODO: Compute the output from each head 
        # and concatenate the results
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, num_heads*head_size) -> all heads concatenated rowwise

        # TODO: Compute the final embedding with the output weight matrix
        out = self.proj(out)  # (B, T, n_embd) -> project down to get again n_embedd as dimenson for each token

        # TODO: Pass the transformer embedding through the dropout layer 
        out = self.dropout(out) # randomly hide some elements during training for regularization
        #print("mha", out)
        return out



c) Feed Forward Layer

In [110]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        # TODO: Create the feedforward layer
        # Hint: use 2 linear layers
        # For the first use ReLU activation 
        # For the second use dropout
        # The hidden dimension should 4 times the input dimension 
        hidden_dim = 4 * n_embd  # hidden dimension 4x input embedding dim
        self.net = nn.Sequential(
            nn.Linear(n_embd, hidden_dim),  # 1. lineare Schicht
            nn.ReLU(),                      # 2. nichtlineare Aktivierung
            nn.Linear(hidden_dim, n_embd),  # 3. lineare Schicht zurück auf n_embd
            nn.Dropout(dropout)              # 4. Dropout zur Regularisierung
        )

    def forward(self, x):
        # TODO: pass the input through the feedforward network you define above 
        x = self.net(x)
        #print("ffw", x)
        return x
    
  


d) Transformer Block

In [111]:
class Block(nn.Module):
    """ The self attention block (encoder) """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        # TODO: Compute the size of each head using n_embd and n_head
        head_size = n_embd // n_head 
        # TODO: Create a multihead attention block with n_head heads of size head_size
        self.sa = MultiHeadAttention(num_heads=n_head, head_size=head_size)
        # TODO: Create a feedforward layer
        self.ffwd = FeedFoward(n_embd= n_embd)
        # TODO: Create two normalization layers of size n_embd
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # TODO: Pass the input x through the first normalization layer
        # TODO: Use the multi-head attention
        # Hint do not forget to add the residual connection
        x = x = x + self.sa(self.ln1(x))
        # TODO: Normalize the output above
        # TODO: Use the feedforward layer 
        # Hint: do not forget to add the residual connection 
        x = x = x + self.ffwd(self.ln2(x))
        #print(x)
        return x


e) MyGPT

In [112]:
class MyGPT(nn.Module):
    """A minimal GPT-style language model"""
    def __init__(self):
        super().__init__()
        # We will define n_embd, vocab_size, block_size, n_layer
        # later as hyperparameters.
        # Each token directly reads off the logits for the next token from a lookup table
        # Input token embedding
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # Positional encoding embedding
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # Stacked encoder blocks
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)]) # various blocks in sequence
        # Final layer normalization
        self.ln_f = nn.LayerNorm(n_embd) 
        # Final linear layer
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        B, T = idx.shape

        # Create the token and positional embeddings
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        # TODO: Create the input embedding for the multi-head attention 
        x = tok_emb + pos_emb 
        # TODO: Use the multi-head attention blocks
        x = self.blocks(x)
        # TODO: Use normalization
        x = x = self.ln_f(x)
        # TODO: Get the token logits using the final linear layer 
        logits = self.lm_head(x)

        if targets is None:
            loss = None
            print("No targets provided, skipping loss computation.")
        else:
            # Compute loss for training
            B, T, C = logits.shape
            #print(logits)
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

## MyGPT Zusammanfassung

## Logits am Ende des Transformers / GPT-Modells

- Am Ende des Transformers hat jeder Token einen Feature-Vektor: (B, T, n_embd)
- Der Linear Layer `lm_head` projiziert diesen Vektor auf die Vokabulargröße: (B, T, vocab_size)
  - Gewichtsmatrix W: (n_embd, vocab_size)
  - Bias b: (vocab_size,)
- Die Ausgaben dieses Layers heißen **Logits**:
  - Rohwerte, die anzeigen, wie stark jedes Wort im Vokabular als nächstes Token passen würde
  - **Keine Wahrscheinlichkeiten**, können negativ sein
- Um Wahrscheinlichkeiten zu erhalten, wendet man Softmax an:
  - P(token_i) = exp(logit_i) / sum_j exp(logit_j)
- Während Training:
  - `F.cross_entropy(logits, targets)` wird verwendet
  - Berechnet intern Softmax + Loss
- Zusammengefasst:
  - Logits = rohe Scores pro Wort
  - Softmax = Wahrscheinlichkeiten
  - Linear Layer = projiziert kontextuelles Embedding auf Vokabular


idx (B, T)
   │
   ▼
Token Embedding (B, T, n_embd)
   │
   ▼
+ Positional Embedding (B, T, n_embd)
   │
   ▼
x (B, T, n_embd)
   │
   ├─► Block 1: Attention + FeedForward + Residual + LayerNorm
   │
   ├─► Block 2: Attention + FeedForward + Residual + LayerNorm
   │
   └─► ... weitere Blocks ...
           │
           ▼
Final LayerNorm ln_f(x) (B, T, n_embd)
           │
           ▼
Final Linear Layer lm_head
           │
           ▼
logits (B, T, vocab_size)
           │
           ▼
Optional Cross-Entropy Loss (falls targets vorhanden)



# Task 3: Generating Text from Language Models

a) Construct the training data and train the model

Let's first define our model and training hyperparameters

In [113]:
# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 7000
eval_interval = 100
learning_rate = 1e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0

torch.manual_seed(1337)

Let's prepare the training data and define our toy tokenizer. For this step make sure that the file path for `tutorial3.txt` is correct.

In [114]:
# TODO: read the training data and store them in a variable 'text'
with open("tutorial3.txt", "r", encoding="utf-8") as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers (our toy tokenizer)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# TODO: Split your data into a training (90%) and validation set
# Hint: before slitting encode the data and store them in a tensor
# Train and validation splits
# encode the entire text as a tensor
data = torch.tensor(encode(text), dtype=torch.long)
# split into training (90%) and validation (10%) sets
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


The loss function we will use during training

In [115]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            #print(Y)
            logits, loss = model(X, targets = Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

Let's train our model! (In google colab this takes around 15min!)

In [ ]:
model = MyGPT()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, targets =yb)
    optimizer.zero_grad()
    loss.backward()
    max_grad_norm = 1.0 # Ein typischer Wert
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
    optimizer.step()

0.214373 M parameters
step 0: train loss 4.8571, val loss 4.8553
step 100: train loss 3.6032, val loss 3.5460
step 200: train loss 3.3269, val loss 3.2853
step 300: train loss 3.1467, val loss 3.1140
step 400: train loss 2.9913, val loss 2.9655
step 500: train loss 2.8870, val loss 2.8701
step 600: train loss 2.7982, val loss 2.7848
step 700: train loss 2.7401, val loss 2.7300
step 800: train loss 2.6908, val loss 2.6918
step 900: train loss 2.6576, val loss 2.6591
step 1000: train loss 2.6263, val loss 2.6317
step 1100: train loss 2.6013, val loss 2.6161
step 1200: train loss 2.5763, val loss 2.5992
step 1300: train loss 2.5617, val loss 2.5816
step 1400: train loss 2.5442, val loss 2.5662
step 1500: train loss 2.5303, val loss 2.5558
step 1600: train loss 2.5164, val loss 2.5347
step 1700: train loss 2.5019, val loss 2.5124


The code below saves the weights for your model so that you do not need to retrain it from scratch

In [ ]:
# Specify the file path to store the model weights
model_weights_file = 'model_weights.pth'  # Or any suitable location

# Function to save model weights
def save_model_weights(model, filepath):
    torch.save(model.state_dict(), filepath)

# Function to load model weights
def load_model_weights(model, filepath):
    model.load_state_dict(torch.load(filepath, weights_only=True))
    model.eval() # Set to evaluation mode

# Save the model's weights if you do not want to retrain the model every time
save_model_weights(m, model_weights_file)
print(f"Model weights saved to {model_weights_file}")


Model weights saved to model_weights.pth


b) Generate text with greedy decoding

Define the function that generates text using greedy decoding

In [ ]:
def generate_greedy(model, idx, max_new_tokens):
  """
    Generate text using greedy sampling.

    Args:
        model: the GPT model
        idx: (B, T) tensor of current token indices
        max_new_tokens: how many new tokens to generate
    Returns:
        idx: (B, T + max_new_tokens) tensor with generated sequence
    """
  # idx is (B, T) array of indices in the current context
  for _ in range(max_new_tokens):
      # crop idx to the last block_size tokens
      idx_cond = idx[:, -block_size:]
      # get the predictions
      logits, loss = model(idx_cond)
      # focus only on the last time step
      logits = logits[:, -1, :] # becomes (B, C)
      # TODO: apply softmax to get probabilities
      probs = F.softmax(logits, dim=-1)  # (B, vocab_size)
      # TODO: sample the index of the token with the highest probability
      idx_next = torch.argmax(probs, dim=-1, keepdim=True)  # (B, 1)
      # append sampled index to the running sequence
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
  return idx

Generate text with greedy decoding

In [ ]:
# Create an instance of the model architecture
new_model = MyGPT() # Uncomment this line to avoid retraining your model
new_m = new_model.to(device) # Uncomment this line to avoid retraining your model

# Load the saved weights after training into the new model
load_model_weights(new_m, model_weights_file) # Uncomment this line to avoid retraining your model

# You can now use the new model
# new_m as you would use the model m
# right after training
m = new_m # Uncomment this line to avoid retraining your model

# TODO: Write your input text
input_text = "Once upon a time"  # Beispiel-Text

# TODO: Encode the input text to 'tokens'
encoded_input = encode(input_text)  # Liste von Integer-IDs

# Create the input tensor and fix the dimensions
context = torch.tensor(encoded_input, device=device, dtype=torch.long).unsqueeze(0)  # (1, T)

# TODO: Generate the encoded response using greedy decoding
max_new_tokens = 100  # Anzahl der zu generierenden Tokens
encoded_output = generate_greedy(m, context, max_new_tokens)  # (1, T + max_new_tokens)

# TODO: Decode the output into text
decoded_output = decode(encoded_output[0].tolist())

print(decoded_output)

Once upon a time																																																																																																				


## Task 4: Better decoding strategies

a) Top-k sampling

In [ ]:
def generate_topk(model, idx, max_new_tokens, topk=5):
  """
    Generates text using the top-k sampling method from a trained neural network model.

    This function starts with a given initial string, then continues to generate text
    for a specified number of characters. It uses top-k sampling for predictions,
    where each next character is chosen from the top k most likely next characters
    as predicted by the neural network model.
    """
  # idx is (B, T) array of indices in the current context
  for _ in range(max_new_tokens):
      # crop idx to the last block_size tokens
      idx_cond = idx[:, -block_size:]
      # get the predictions
      logits, loss = model(idx_cond)
      # focus only on the last time step
      logits = logits[:, -1, :] # becomes (B, C)
      # TODO: apply softmax to get probabilities
      probs = 
      # TODO: get the top-k probabilities and their indices
      probs_topk, indices_topk = 
      # TODO: normalize the top-k probabilities
      probs_topk_normalized = 
      # TODO: sample the index of the next token among the top-k tokens
      idx_next_in_topk = 
      # TODO: get the index of the next token from the vocabulary
      # Hint: make sure that idx_next has dimensions (B, 1) 
      idx_next = 
      # append sampled index to the running sequence
      idx = torch.cat((idx, idx_next), dim=1) # size (B, T+1)
  return idx

Generate text with top-k sampling

In [ ]:
# Create an instance of the model architecture
# new_model = MyGPT() # Uncomment this line to avoid retraining your model
# new_m = new_model.to(device) # Uncomment this line to avoid retraining your model

# Load the saved weights after training into the new model
# load_model_weights(new_m, model_weights_file) # Uncomment this line to avoid retraining your model

# You can now use the new model
# new_m as you would use the model m
# right after training
# m = new_m # Uncomment this line to avoid retraining your model

# TODO: Write your input text
input_text = 
# TODO: Encode the input text to 'tokens'
encoded_input = 
# Create the input tensor and fix the dimensions
context = torch.tensor(encoded_input, device=device, dtype=torch.long).unsqueeze(0)

# Define the k value
topk = 5 # Try different values
# TODO: Generate the encoded response using top-k sampling
encoded_output = 
# TODO: Decode the output into text
decoded_output = 
print(decoded_output)

b) Temperature sampling

In [ ]:
def generate_topk_temperature(model, idx, max_new_tokens, topk=5, T=0.8):
  # Avoid division with 0
  if T == 0:
    return generate_greedy(model, idx, max_new_tokens)
  # idx is (B, T) array of indices in the current context
  for _ in range(max_new_tokens):
      # crop idx to the last block_size tokens
      idx_cond = idx[:, -block_size:]
      # get the predictions
      logits, loss = model(idx_cond)
      # focus only on the last time step
      logits = logits[:, -1, :] # becomes (B, C)
      # TODO: implement temperature topk sampling
      logits =
      # TODO: apply softmax to get probabilities
      probs = 
      # TODO: get the top-k probabilities and their indices
      probs_topk, indices_topk = 
      # TODO: normalize the top-k probabilities
      probs_topk_normalized = 
      # TODO: sample the index of the next token among the top-k tokens
      idx_next_in_topk = 
      # TODO: get the index of the next token from the vocabulary
      # Hint: make sure that idx_next has dimensions (B, 1) 
      idx_next = 
      # append sampled index to the running sequence
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
  return idx

Generate text using topk and temperature sampling

In [ ]:
# Create an instance of the model architecture
# new_model = MyGPT() # Uncomment this line to avoid retraining your model
# new_m = new_model.to(device) # Uncomment this line to avoid retraining your model

# Load the saved weights after training into the new model
# load_model_weights(new_m, model_weights_file) # Uncomment this line to avoid retraining your model

# You can now use the new model
# new_m as you would use the model m
# right after training
# m = new_m # Uncomment this line to avoid retraining your model

# TODO: create and encode the input as above
input_text = 
# TODO: Encode the input text to 'tokens'
encoded_input = 
# Create the input tensor and fix the dimensions
context = torch.tensor(encoded_input, device=device, dtype=torch.long).unsqueeze(0)

T = 0.8 # Try different values
topk = 5 # the k-value for top-k
# TODO: Generate the response using top-k with temperature sampling
encoded_output = 
# TODO: Decode the output into text
decoded_output = 
print(decoded_output)

c) Beam search

In [ ]:
def generate_beam_search(model, idx, max_new_tokens, beam_width=5, T=0.5):
    """
    Generates text using the beam search algorithm with a trained neural network model.

    This function starts with a given initial string and uses beam search to generate a text
    sequence of a specified size. Beam search involves maintaining a set of the most promising
    sequences at each step and expanding them to find the most likely overall sequence.
    """
    # idx is (B, T) array of indices in the current context
    # TODO: implement the beam search algorithm

    idx_next = 
    idx = torch.cat((idx, idx_next), dim=1) # (B, T+seq)
    return idx


Generate text with beam search

In [ ]:
# Create an instance of the model architecture
# new_model = MyGPT() # Uncomment this line to avoid retraining your model
# new_m = new_model.to(device) # Uncomment this line to avoid retraining your model

# Load the saved weights after training into the new model
# load_model_weights(new_m, model_weights_file) # Uncomment this line to avoid retraining your model

# You can now use the new model
# new_m as you would use the model m
# right after training
# m = new_m # Uncomment this line to avoid retraining your model

# TODO: Write your input text
input_text = 
# TODO: Encode the input text to 'tokens'
encoded_input = 
# Create the input tensor and fix the dimensions
context = torch.tensor(encoded_input, device=device, dtype=torch.long).unsqueeze(0)

beam_width = 2 # Try different values
# TODO: Generate the encoded response using beam search 
encoded_output = 

# TODO: Decode the output into text
decoded_output = 
print(decoded_output)